# Notebook_C (Sinh viên A): WOMD v1.2 — dữ liệu, EDA và SQL

Notebook này mô tả **đúng mẫu WOMD v1.2 mà Notebook A đã tạo và Notebook B dùng để huấn luyện**: đọc `data/processed/womd_v1_2.parquet`, trong đó yaw rate đã xử lý góc nhảy qua ±π và gia tốc, yaw rate đã cắt ngưỡng nhiễu. Hai file cũ `primary_waymo.parquet` và `secondary_waymo.parquet` là một mẫu khác (yaw rate chưa xử lý góc), nên không dùng nữa.

Hai "feature views" được tách từ cùng bảng này:

- **Main / motion view**: `unit`, `ts`, `speed`
- **Auxiliary / context view**: `unit`, `ts`, `acc`, `yaw_rate`, `n_agents`, `length`

**Notebook_C không bàn giao dữ liệu.** `feat.parquet` và `manifest.json` do Notebook A tạo, Notebook C chỉ đọc. Mọi bảng và hình của C nằm trong `report/C/` để không ghi đè kết quả của Notebook A và B. Số liệu và hình RQ1 đưa vào bài lấy từ Notebook A (`paper_numbers_A.csv`, Fig. 2). Notebook C dùng để kiểm chéo bằng SQL và EDA mở rộng.

**Ba câu hỏi nghiên cứu dùng cho nhánh WOMD v1.2:**

* **RQ1 (dữ liệu và mô tả)**: Chuyển động xe thay đổi như thế nào theo mật độ cảnh, cường độ rẽ, tiến trình quỹ đạo và các feature context?
* **RQ2 (mô hình và tổng quát hoá)**: Mô hình nào tốt nhất cho 3 s và 8 s trên các scenario chưa thấy? (Notebook B)
* **RQ3 (tin cậy và vận hành)**: Khoảng conformal có bao phủ đúng không, cảnh báo hard case ở mức nào? (Notebook B)

In [1]:
# CONFIG: WOMD v1.2, the SAME sample that Notebook A prepared and Notebook B models
TOPIC = "Dự đoán quãng đường xe có khoảng dự báo conformal trên Waymo Open Motion Dataset"
GROUP = "Nhom 7"
WOMD_SOURCE = {
    "name": "Waymo Open Motion Dataset v1.2",
    "url": "https://waymo.com/open/data/motion/",
    "license": "Waymo Open Dataset License",
    "path": "data/processed/womd_v1_2.parquet",
}
PRIMARY_VIEW = {"name": "motion/target feature view", "path": WOMD_SOURCE["path"]}
AUX_VIEW = {"name": "vehicle/scene context feature view", "path": WOMD_SOURCE["path"]}
OUT = "report/C"  # all Notebook C outputs go here so they never overwrite Notebook A/B files
UNIT_COL, TIME_COL, TARGET_COL = "unit", "ts", "speed"
EXOG_COLS = ["acc", "yaw_rate", "n_agents", "length"]
FREQ = "100ms"
HORIZONS = [30, 80]  # 30 steps = 3 s, 80 steps = 8 s at 10 Hz
HISTORY_STEPS = 10  # 10 steps = 1 s of history
EVENT_QUANTILE = 0.9
TURN_EPS = 1e-6  # |yaw rate| at or below this = no heading change (group 0), as in Notebook A
print("Topic:", TOPIC)
print("Group:", GROUP)
print("Dataset:", WOMD_SOURCE["name"], "| file:", WOMD_SOURCE["path"])

Topic: Dự đoán quãng đường xe có khoảng dự báo conformal trên Waymo Open Motion Dataset
Group: Nhom 7
Dataset: Waymo Open Motion Dataset v1.2 | file: data/processed/womd_v1_2.parquet


In [2]:
# AI Audit Log helper: every prompt that changed your work is one row (2 minutes per entry, 3-5 per week)
import pandas as pd, os, datetime as dt
os.makedirs("report/C", exist_ok=True)
AUDIT_PATH = "report/C/ai_audit_log.csv"
def audit(step, prompt, tool, ai_output_summary, verified_how, decision, hallucination=False):
    """Append one entry. decision: what you kept / changed / rejected. hallucination=True if the AI answer was wrong and you caught it."""
    row = {
        "date": dt.date.today().isoformat(),
        "group": GROUP,
        "step": step,
        "prompt": prompt[:500],
        "tool": tool,
        "ai_output": ai_output_summary[:500],
        "verified_how": verified_how[:300],
        "decision": decision[:300],
        "hallucination": int(hallucination),
    }
    df = pd.DataFrame([row])
    df.to_csv(AUDIT_PATH, mode="a", header=not os.path.exists(AUDIT_PATH), index=False)
    print("audit entry saved:", step)
# example (delete after reading): audit("Step 2", "Given columns ... how to treat gaps longer than 3 steps?", "Claude", "suggested interpolate(limit=3) then drop", "checked share of gaps > 3 in Q4 below", "kept limit=3, dropped 1.2% rows")

## Hướng dẫn hỏi AI

Quy tắc: hỏi **cụ thể** (kèm tên cột, kích thước, thông báo lỗi), yêu cầu AI **giải thích lý do** và **nêu cách kiểm tra**, rồi tự kiểm tra trước khi dùng. Mỗi prompt làm thay đổi bài phải ghi vào AI Audit Log bằng hàm `audit(...)`.

| Bước | Mẫu prompt | Cần tự kiểm tra gì |
|---|---|---|
| Kiểm tra feature views | "Hai file của tôi đều được trích từ WOMD v1.2: main view có `unit, ts, speed`, auxiliary view có `unit, ts, acc, yaw_rate, n_agents, length`. Hãy đề xuất kiểm tra để xác nhận hai view cùng key và không tạo rò rỉ khi ghép." | Kiểm tra số dòng, duplicate key và exact key match |
| Làm sạch | "Cột {cột} có {x}% thiếu, thiếu theo cụm dài nhất {n} bước. Nên nội suy hay bỏ? Giải thích rủi ro rò rỉ tương lai." | Kiểm tra nội suy không dùng dữ liệu tương lai cho feature dự báo |
| SQL | "Viết truy vấn DuckDB tạo feature lịch sử 1 giây và rolling mean theo `unit`, không dùng dữ liệu tương lai." | Kiểm tra NULL ở đầu chuỗi và alignment theo unit |
| EDA | "Từ bảng thống kê sau, nêu 3 nhận xét có thể kiểm chứng về speed, scene density và turning." | Mỗi nhận xét phải chỉ ra ô số liệu tương ứng |
| Lỗi | "Lỗi: {dán nguyên văn}. Ngữ cảnh: {ô code}. Nguyên nhân và cách sửa tối thiểu?" | Sửa xong chạy lại test |

Ví dụ Audit Log: ghi rõ prompt, câu trả lời AI, cách bạn kiểm tra trên dữ liệu thật và quyết định cuối cùng.


## Bước 0: môi trường và cấu trúc thư mục

**Làm gì**: tạo môi trường ảo một lần, cài thư viện, tạo thư mục chuẩn. **Vì sao**: cả hai sinh viên và giảng viên chạy cùng phiên bản thư viện thì kết quả tái tạo được; `requirements.txt` là bằng chứng.

**Cấu trúc**: `data/raw` (file gốc, không sửa), `data/processed` (parquet sạch), `sql/` (truy vấn), `report/` (bảng, hình, nhật ký), `notebooks/`.

In [3]:
# Step 0a: run once in a terminal (not in the notebook)
# python -m venv .venv && source .venv/bin/activate      (Windows: .venv\Scripts\activate)
# pip install pandas numpy duckdb pyarrow scikit-learn lightgbm statsmodels matplotlib seaborn ydata-profiling mapie shap requests
# pip freeze > requirements.txt
import os
for d in ["data/raw", "data/processed", "sql", "report", "report/C", "notebooks"]:
    os.makedirs(d, exist_ok=True)
print("folders ready")

folders ready


In [4]:
# Step 0b: imports and figure style (same paper style as Notebooks A and B: print width, 8 pt, white, one accent colour)
import pandas as pd, numpy as np, duckdb, json, hashlib, warnings, re, os
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
np.random.seed(42)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
FIG_W = 4.8  # LNCS text width in inches; figures are drawn at print size so 8 pt stays 8 pt
C_TEXT, C_BASE, C_MID, C_ACCENT, C_GREY = "#1F3B57", "#9CC9E8", "#4F8FC0", "#FF7F66", "#AEB9C4"
plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 8,
        "axes.labelsize": 8,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "text.color": C_TEXT,
        "axes.labelcolor": C_TEXT,
        "axes.edgecolor": C_TEXT,
        "xtick.color": C_TEXT,
        "ytick.color": C_TEXT,
        "axes.linewidth": 0.6,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "axes.grid": False,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "savefig.dpi": 300,
        "pdf.fonttype": 42,
    }
)
TEAL, ACC, GREY = C_MID, C_ACCENT, C_BASE  # old colour names kept for the exercise cells
NICE = {
    "speed": "Speed (m/s)",
    "acc": "Acceleration (m/s²)",
    "yaw_rate": "Yaw rate (rad/s)",
    "n_agents": "Agents in scene",
    "length": "Vehicle length (m)",
}
def style(ax):
    ax.spines[["top", "right"]].set_visible(False)
def panel(ax, lab):
    ax.set_title(
        lab, loc="left", fontsize=8, pad=3
    )  # panel label only; the explanation belongs in the caption
def savefig(fig, name):
    fig.tight_layout(pad=0.4)
    fig.savefig(f"{OUT}/{name}.png")
    plt.close(fig)
    print(f"saved {OUT}/{name}.png")
con = duckdb.connect()
print("pandas", pd.__version__, "duckdb", duckdb.__version__)

pandas 2.3.3 duckdb 1.5.5


## Bước 1: nguồn dữ liệu và hai feature views

Notebook_C dùng **một nguồn duy nhất: WOMD v1.2**, đọc từ bảng `data/processed/womd_v1_2.parquet` mà Notebook A đã lấy mẫu (1,200 xe, 20 shard) và tính động học. Hai view chỉ là hai phép chiếu cột từ cùng bảng này:

- main/motion view: `unit`, `ts`, `speed`
- auxiliary/context view: `unit`, `ts`, `acc`, `yaw_rate`, `n_agents`, `length`

**Thời gian `ts`.** Notebook A giữ timestamp thật (lệch tối đa vài phần trăm giây so với 10 Hz) và đã kiểm tra độ lệch đó khi tạo mục tiêu 3 s và 8 s. Ở đây `ts` được đặt theo lưới danh nghĩa `2000-01-01 + step × 100 ms`, chỉ để mã hoá tiến trình quỹ đạo cho EDA; không diễn giải thành giờ hay ngày.

In [5]:
# Step 1a: write the single public-source registry entry
SOURCES_PATH = "report/C/data_sources.csv"
def write_source_registry(src, rows=None, cols=None, start=None, end=None, note=""):
    row = {
        **src,
        "downloaded": pd.Timestamp.today().date().isoformat(),
        "rows": rows,
        "cols": cols,
        "start": start,
        "end": end,
        "note": note,
    }
    pd.DataFrame([row]).to_csv(SOURCES_PATH, index=False)
    return row
print("registry at", SOURCES_PATH)

registry at report/C/data_sources.csv


In [6]:
# Step 1b: load Notebook A's WOMD v1.2 table and build the main/motion view
def load_table(path):
    """Read csv/parquet with DuckDB and return a pandas DataFrame."""
    if path.endswith(".parquet"):
        return con.execute(f"SELECT * FROM read_parquet('{path}')").df()
    return con.execute(f"SELECT * FROM read_csv_auto('{path}', union_by_name=true, sample_size=-1)").df()
src = load_table(WOMD_SOURCE["path"])
src[TIME_COL] = pd.Timestamp("2000-01-01") + pd.to_timedelta(
    src["step"] * 100, unit="ms"
)  # nominal 10 Hz grid
raw = src[[UNIT_COL, TIME_COL, TARGET_COL]].copy()
print(raw.shape, "| vehicles:", raw[UNIT_COL].nunique(), "| scenarios:", src["scenario"].nunique())
display(raw.head(3))
write_source_registry(
    WOMD_SOURCE,
    rows=len(raw),
    cols=raw.shape[1],
    start=str(raw[TIME_COL].min()),
    end=str(raw[TIME_COL].max()),
    note="One WOMD v1.2 sample prepared by Notebook A (womd_v1_2.parquet); main and auxiliary views are column projections of it.",
)

(109200, 3) | vehicles: 1200 | scenarios: 1073


,unit,ts,speed
0,1038e39306f1e9e2_329,2000-01-01 00:00:00.000,0.0
1,1038e39306f1e9e2_329,2000-01-01 00:00:00.100,0.0
2,1038e39306f1e9e2_329,2000-01-01 00:00:00.200,0.0


{'name': 'Waymo Open Motion Dataset v1.2',
 'url': 'https://waymo.com/open/data/motion/',
 'license': 'Waymo Open Dataset License',
 'path': 'data/processed/womd_v1_2.parquet',
 'downloaded': '2026-10-11',
 'rows': 109200,
 'cols': 3,
 'start': '2000-01-01 00:00:00',
 'end': '2000-01-01 00:00:09',
 'note': 'One WOMD v1.2 sample prepared by Notebook A (womd_v1_2.parquet); main and auxiliary views are column projections of it.'}

In [7]:
# Step 1c: auxiliary/context view from the same WOMD v1.2 records
sec = src[[UNIT_COL, TIME_COL] + EXOG_COLS].copy()
key_cols = [UNIT_COL, TIME_COL]
raw_keys = raw[key_cols].sort_values(key_cols).reset_index(drop=True)
sec_keys = sec[key_cols].sort_values(key_cols).reset_index(drop=True)
print(sec.shape)
display(sec.head(3))
print("same number of rows:", len(raw) == len(sec))
print("exact key match:", raw_keys.equals(sec_keys))
print("primary duplicate keys:", raw.duplicated(key_cols).sum())
print("auxiliary duplicate keys:", sec.duplicated(key_cols).sum())
assert raw_keys.equals(sec_keys), "The two WOMD feature views are not aligned on [unit, ts]."
# yaw rate must be wrapped at ±pi: an unwrapped heading jump would show up as about 2*pi/0.1 = 62.8 rad/s
print(
    "max |yaw rate| (rad/s):",
    round(float(sec["yaw_rate"].abs().max()), 3),
    "| max |acc| (m/s^2):",
    round(float(sec["acc"].abs().max()), 3),
)
assert sec["yaw_rate"].abs().max() < 2 * np.pi / 0.1 - 1, "yaw rate looks unwrapped"

(109200, 6)


,unit,ts,acc,yaw_rate,n_agents,length
0,1038e39306f1e9e2_329,2000-01-01 00:00:00.000,NaN,NaN,12,4.948853
1,1038e39306f1e9e2_329,2000-01-01 00:00:00.100,0.0,0.0,12,4.948853
2,1038e39306f1e9e2_329,2000-01-01 00:00:00.200,0.0,0.0,12,4.948853


same number of rows: True
exact key match: True
primary duplicate keys: 0
auxiliary duplicate keys: 0
max |yaw rate| (rad/s): 2.0 | max |acc| (m/s^2): 10.0


### Xuất và đọc lại dữ liệu bằng CSV

**Quy ước file**: mỗi bước lưu một file CSV để giảng viên và bạn cùng nhóm mở được bằng Excel. Các file là `data/processed/C_primary_snapshot.csv` (main view), `data/processed/C_clean.csv` (sau làm sạch). Tên có tiền tố `C_` để không trùng với file của Notebook A. Khi đọc lại CSV, luôn ép kiểu thời gian bằng `parse_dates` và kiểm tra số dòng.

In [8]:
# Export the standardised raw snapshot to CSV and read it back to confirm the round trip (dtypes and row count)
RAW_CSV = "data/processed/C_primary_snapshot.csv"
raw.to_csv(RAW_CSV, index=False)
check = pd.read_csv(RAW_CSV, parse_dates=[TIME_COL])
assert len(check) == len(raw), "row count changed in the CSV round trip"
print("wrote", RAW_CSV, "| rows", len(check), "| columns", list(check.columns)[:8])

wrote data/processed/C_primary_snapshot.csv | rows 109200 | columns ['unit', 'ts', 'speed']


### Bước 1d: kiểm tra phạm vi dữ liệu

Notebook A đã lấy đủ 20 shard WOMD v1.2 vào một bảng, nên không còn phải ghép nhiều chunk. Ô dưới chỉ in phạm vi dữ liệu.

In [9]:
# Step 1d: data scope (Notebook A already merged all 20 shards into one table)
steps = src.groupby(UNIT_COL)["step"].agg(["min", "max", "count"])
print(
    "vehicles:",
    raw[UNIT_COL].nunique(),
    "| scenarios:",
    src["scenario"].nunique(),
    "| steps per vehicle:",
    steps["count"].unique().tolist(),
)
print("step range:", int(steps["min"].min()), "to", int(steps["max"].max()))

vehicles: 1200 | scenarios: 1073 | steps per vehicle: [91]
step range: 0 to 90


## Bước 2: đối chiếu hai feature views và làm sạch có nhật ký

Vì main view và auxiliary view đến từ **cùng WOMD v1.2**, bước kiểm tra đúng không phải là "hai nguồn có nhất quán không" mà là:

1. hai view có cùng số dòng hay không;
2. key `[unit, ts]` có khớp chính xác hay không;
3. có duplicate key hay không;
4. tỷ lệ missing của từng auxiliary feature là bao nhiêu.

Sau đó làm sạch có nhật ký: mỗi thao tác ghi số dòng trước/sau và lý do để report có thể tái tạo.


In [10]:
# Step 2a: audit alignment between two feature views from the same WOMD v1.2 source
key_cols = [UNIT_COL, TIME_COL]
main_keys = raw[key_cols].sort_values(key_cols).reset_index(drop=True)
aux_keys = sec[key_cols].sort_values(key_cols).reset_index(drop=True)
rows = [
    ["public dataset", WOMD_SOURCE["name"]],
    ["main-view rows", len(raw)],
    ["auxiliary-view rows", len(sec)],
    ["main duplicate keys", int(raw.duplicated(key_cols).sum())],
    ["auxiliary duplicate keys", int(sec.duplicated(key_cols).sum())],
    ["exact [unit, ts] key match", bool(main_keys.equals(aux_keys))],
    ["main earliest timestamp", raw[TIME_COL].min()],
    ["main latest timestamp", raw[TIME_COL].max()],
]
for c in EXOG_COLS:
    if c in sec.columns:
        rows.append([f"{c} missing share", round(float(sec[c].isna().mean()), 4)])
view_audit = pd.DataFrame(rows, columns=["metric", "value"])
view_audit.to_csv("report/C/table_feature_view_alignment.csv", index=False)
display(view_audit)
assert main_keys.equals(aux_keys), "WOMD feature views do not align exactly."

,metric,value
0,public dataset,Waymo Open Motion Dataset v1.2
1,main-view rows,109200
2,auxiliary-view rows,109200
3,main duplicate keys,0
4,auxiliary duplicate keys,0
5,"exact [unit, ts] key match",True
6,main earliest timestamp,2000-01-01 00:00:00
7,main latest timestamp,2000-01-01 00:00:09
8,acc missing share,0.011
9,yaw_rate missing share,0.011


In [11]:
# Step 2b: cleaning with a log; each step records rows before and after and the reason
clean_log = []
def step(df, name, fn, reason):
    n0 = len(df)
    out = fn(df)
    clean_log.append([name, n0, len(out), n0 - len(out), reason])
    return out
df = raw.copy()
df = step(df, "parse time", lambda x: x.dropna(subset=[TIME_COL]), "rows with unparseable timestamps")
df = step(df, "drop duplicates", lambda x: x.drop_duplicates([UNIT_COL, TIME_COL]), "same unit and timestamp")
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = step(df, "drop missing target", lambda x: x.dropna(subset=[TARGET_COL]), "target missing")
df = step(df, "physical range", lambda x: x[x[TARGET_COL] >= 0], "speed must be non-negative")
pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]).to_csv(
    "report/C/table_cleaning_log.csv", index=False
)
display(pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]))

,step,rows_before,rows_after,rows_removed,reason
0,parse time,109200,109200,0,rows with unparseable timestamps
1,drop duplicates,109200,109200,0,same unit and timestamp
2,drop missing target,109200,109200,0,target missing
3,physical range,109200,109200,0,speed must be non-negative


In [12]:
# Step 2c: regular time grid per unit, then short-gap interpolation (limit = 3 steps, never across long gaps)
df = (
    df.set_index(TIME_COL)
    .groupby(UNIT_COL)[
        [TARGET_COL]
        + [
            c
            for c in df.columns
            if c not in (UNIT_COL, TIME_COL, TARGET_COL) and pd.api.types.is_numeric_dtype(df[c])
        ]
    ]
    .resample(FREQ)
    .mean()
    .reset_index()
)
gap = df.groupby(UNIT_COL)[TARGET_COL].apply(lambda s: s.isna().mean()).rename("missing_share")
df[TARGET_COL] = df.groupby(UNIT_COL)[TARGET_COL].transform(lambda s: s.interpolate(limit=3))
n_before = len(df)
df = df.dropna(subset=[TARGET_COL])
clean_log.append(
    [
        "regular grid + interpolate(limit=3)",
        n_before,
        len(df),
        n_before - len(df),
        "gaps longer than 3 steps dropped",
    ]
)
print("missing share per unit before interpolation:")
display(gap.describe().round(3))

missing share per unit before interpolation:


count    1200.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: missing_share, dtype: float64

In [13]:
# Step 2d: keep units with enough history (at least 90% of the longest unit) so the global model is not dominated by fragments
cnt = df.groupby(UNIT_COL).size()
keep = cnt[cnt >= 0.9 * cnt.max()].index
n_before = len(df)
df = df[df[UNIT_COL].isin(keep)]
clean_log.append(
    [
        "drop short units",
        n_before,
        len(df),
        n_before - len(df),
        f"{len(cnt) - len(keep)} units with < 90% coverage",
    ]
)
print("units kept:", len(keep), "of", len(cnt))

units kept: 1200 of 1200


In [14]:
# Step 2e: exact key merge of auxiliary features from the same WOMD v1.2 records
# Do NOT use nearest-time merge_asof here: both views were extracted from the same [unit, ts] rows.
key_cols = [UNIT_COL, TIME_COL]
aux_cols = [c for c in EXOG_COLS if c in sec.columns]
sec_num = sec[key_cols + aux_cols].copy()
for c in aux_cols:
    sec_num[c] = pd.to_numeric(sec_num[c], errors="coerce")
assert sec_num.duplicated(key_cols).sum() == 0, "duplicate keys in auxiliary view"
# df currently comes from the main view only, so exact one-to-one/one-to-zero matching is expected.
m = df.merge(sec_num, on=key_cols, how="left", validate="one_to_one")
share = {c: round(float(m[c].notna().mean()), 3) for c in aux_cols}
print("share of rows with each auxiliary covariate:", share)
df = m.sort_values([UNIT_COL, TIME_COL]).reset_index(drop=True)

share of rows with each auxiliary covariate: {'acc': 0.989, 'yaw_rate': 0.989, 'n_agents': 1.0, 'length': 1.0}


In [15]:
# Step 2f: save the clean table and cleaning log
df.to_parquet("data/processed/C_clean.parquet", index=False)
df.to_csv("data/processed/C_clean.csv", index=False)
chk = pd.read_csv("data/processed/C_clean.csv", parse_dates=[TIME_COL])
assert len(chk) == len(df)
assert chk[TARGET_COL].notna().all(), "C_clean.csv round trip failed"
print("wrote data/processed/C_clean.csv with", len(chk), "rows")
pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]).to_csv(
    "report/C/table_cleaning_log.csv", index=False
)
print(
    f"Data statement: one source, {WOMD_SOURCE['name']} ({WOMD_SOURCE['license']}), represented by a motion view and an "
    f"auxiliary context view; {len(df):,} rows, {df[UNIT_COL].nunique()} units, frequency {FREQ}; "
    f"{sum(r[3] for r in clean_log):,} rows removed by cleaning."
)

wrote data/processed/C_clean.csv with 109200 rows
Data statement: one source, Waymo Open Motion Dataset v1.2 (Waymo Open Dataset License), represented by a motion view and an auxiliary context view; 109,200 rows, 1200 units, frequency 100ms; 0 rows removed by cleaning.


## EDA (RQ1): phân bố, trajectory progress, scene context và tự tương quan

Mỗi hình phải trả lời một câu hỏi kiểm chứng được. Thứ tự:

1. phân bố của speed;
2. chuỗi theo thời gian của vài trajectory;
3. thay đổi theo **trajectory progress**, không diễn giải `ts` giả lập như thời gian lịch;
4. khác biệt giữa trajectory;
5. quan hệ với auxiliary/context features (`acc`, `yaw_rate`, `n_agents`, `length`);
6. cross-correlation và ACF để chọn feature lịch sử;
7. missingness và scene density.

Không gọi các pattern này là "mùa vụ theo lịch", vì timestamp `2000-01-01 + step × 100 ms` chỉ mã hoá tiến trình quỹ đạo.


In [16]:
# EDA 1: automatic profile (open report/C/profile.html in a browser) and summary statistics
from ydata_profiling import ProfileReport
sample = df.sample(min(20000, len(df)), random_state=42)
ProfileReport(sample, title=f"Profile {GROUP}", minimal=True).to_file("report/C/profile.html")
desc = (
    df[[TARGET_COL] + [c for c in EXOG_COLS if c in df.columns]]
    .describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
    .T.round(3)
)
desc.to_csv("report/C/table_describe.csv")
display(desc)

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 7/7 [00:00<00:00, 301.11it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

,count,mean,std,min,1%,5%,50%,95%,99%,max
speed,109200.0,2.466,4.755,0.000,0.000,0.000,0.000,13.307,19.983,35.881
acc,108000.0,0.032,1.659,-10.000,-5.415,-2.356,0.000,2.463,5.676,10.000
yaw_rate,108000.0,0.001,0.070,-2.000,-0.263,-0.040,0.000,0.041,0.278,2.000
n_agents,109200.0,55.994,36.380,2.000,8.000,14.000,46.000,122.000,180.000,284.000
length,109200.0,4.832,0.805,3.631,4.064,4.242,4.652,5.755,9.046,13.346


In [17]:
# Extreme acceleration and yaw-rate values (Notebook A clipped |acc| and |yaw rate| at physical limits)
for c in ["acc", "yaw_rate"]:
    s = df[c].dropna()
    print(
        "\n",
        c,
        "| n =",
        len(s),
        "| q0.1%, q1%, q99%, q99.9% =",
        s.quantile([0.001, 0.01, 0.99, 0.999]).round(4).to_dict(),
    )
    extreme = df.loc[df[c].abs() >= s.abs().quantile(0.999), [UNIT_COL, TIME_COL, c]]
    print(
        "rows at or above the 99.9% |value|:", len(extreme), "| max |value|:", round(float(s.abs().max()), 3)
    )
    display(extreme.head(10))


 acc | n = 108000 | q0.1%, q1%, q99%, q99.9% = {0.001: -10.0, 0.01: -5.4148, 0.99: 5.6758, 0.999: 10.0}
rows at or above the 99.9% |value|: 574 | max |value|: 10.0


,unit,ts,acc
1558,13970b5053f8de9e_1047,2000-01-01 00:00:01.100,-10.0
1602,13970b5053f8de9e_1047,2000-01-01 00:00:05.500,10.0
1603,13970b5053f8de9e_1047,2000-01-01 00:00:05.600,-10.0
2044,14edb7d422d6c1f8_1954,2000-01-01 00:00:04.200,-10.0
2045,14edb7d422d6c1f8_1954,2000-01-01 00:00:04.300,10.0
2064,14edb7d422d6c1f8_1954,2000-01-01 00:00:06.200,-10.0
2065,14edb7d422d6c1f8_1954,2000-01-01 00:00:06.300,10.0
4868,19f8e7f25f8ab78_7,2000-01-01 00:00:04.500,10.0
4869,19f8e7f25f8ab78_7,2000-01-01 00:00:04.600,-10.0
6464,1dd3f390c0b8e767_2296,2000-01-01 00:00:00.300,10.0



 yaw_rate | n = 108000 | q0.1%, q1%, q99%, q99.9% = {0.001: -0.4925, 0.01: -0.2628, 0.99: 0.278, 0.999: 0.5813}
rows at or above the 99.9% |value|: 108 | max |value|: 2.0


,unit,ts,yaw_rate
4868,19f8e7f25f8ab78_7,2000-01-01 00:00:04.500,0.698501
15112,2cf50e2494074a09_15,2000-01-01 00:00:00.600,0.738746
15423,2d8629af8a5d0abf_762,2000-01-01 00:00:04.400,0.651014
15428,2d8629af8a5d0abf_762,2000-01-01 00:00:04.900,0.619695
15429,2d8629af8a5d0abf_762,2000-01-01 00:00:05.000,0.687657
15430,2d8629af8a5d0abf_762,2000-01-01 00:00:05.100,0.638272
15431,2d8629af8a5d0abf_762,2000-01-01 00:00:05.200,0.654711
15433,2d8629af8a5d0abf_762,2000-01-01 00:00:05.400,0.896295
15435,2d8629af8a5d0abf_762,2000-01-01 00:00:05.600,0.724439
15436,2d8629af8a5d0abf_762,2000-01-01 00:00:05.700,0.735599


In [18]:
# EDA 2: distribution of speed on a linear and a log scale
fig, axes = plt.subplots(1, 2, figsize=(FIG_W, 2.0))
axes[0].hist(df[TARGET_COL], bins=60, color=C_BASE, edgecolor=C_TEXT, linewidth=0.3)
axes[0].set_xlabel(NICE[TARGET_COL])
axes[0].set_ylabel("Rows")
panel(axes[0], "(a)")
pos = df[TARGET_COL][df[TARGET_COL] > 0]
axes[1].hist(np.log1p(pos), bins=60, color=C_BASE, edgecolor=C_TEXT, linewidth=0.3)
axes[1].set_xlabel("log(1 + speed), moving rows")
panel(axes[1], "(b)")
savefig(fig, "fig_eda_distribution")
print(
    "skewness:",
    round(df[TARGET_COL].skew(), 2),
    "| share of rows with speed = 0:",
    round(float((df[TARGET_COL] == 0).mean()), 4),
)

saved report/C/fig_eda_distribution.png
skewness: 2.18 | share of rows with speed = 0: 0.5584


In [19]:
# EDA 3: speed over time for four moving vehicles (25th, 50th, 75th and 95th percentile of mean speed among moving vehicles)
um = df.groupby(UNIT_COL)[TARGET_COL].mean()
um = um[um > 0.5].sort_values()
units = [um.index[int(q * (len(um) - 1))] for q in [0.25, 0.5, 0.75, 0.95]]
fig, ax = plt.subplots(figsize=(FIG_W, 2.2))
ends = []
for k, (u, col) in enumerate(zip(units, [C_BASE, C_MID, C_GREY, C_TEXT])):
    g = df[df[UNIT_COL] == u].sort_values(TIME_COL)
    t = (g[TIME_COL] - g[TIME_COL].min()).dt.total_seconds()
    ax.plot(t, g[TARGET_COL], lw=0.9, color=col)
    ends.append([g[TARGET_COL].iloc[-1], t.iloc[-1], f"vehicle {k + 1}"])
gap = 0.07 * max(e[0] for e in ends)  # keep end labels at least this far apart
ends.sort()
for j in range(1, len(ends)):
    ends[j][0] = max(ends[j][0], ends[j - 1][0] + gap)
for y_lab, t_end, lab in ends:
    ax.text(t_end + 0.1, y_lab, lab, va="center")
ax.set_xlim(0, 10.6)
ax.set_xlabel("Elapsed time (s)")
ax.set_ylabel(NICE[TARGET_COL])
savefig(fig, "fig_eda_timeseries")

saved report/C/fig_eda_timeseries.png


In [20]:
# EDA 4: mean speed over trajectory progress
tmp4 = df.copy()
tmp4["second"] = np.floor(
    (tmp4[TIME_COL] - tmp4.groupby(UNIT_COL)[TIME_COL].transform("min")).dt.total_seconds()
).astype(int)
s = tmp4.groupby("second")[TARGET_COL].mean()
fig, ax = plt.subplots(figsize=(FIG_W, 2.0))
ax.plot(s.index, s.values, marker="o", ms=3, color=C_MID)
ax.set_ylim(0, s.max() * 1.2)
ax.set_xlabel("Elapsed time (s)")
ax.set_ylabel("Mean speed (m/s)")
fig.tight_layout(pad=0.4)
fig.subplots_adjust(left=0.13)  # keep the y-axis label inside the saved figure
fig.savefig(f"{OUT}/fig_eda_trajectory_progress.png")
plt.close(fig)
print(f"saved {OUT}/fig_eda_trajectory_progress.png")
progress_strength = round(s.std() / tmp4[TARGET_COL].std(), 3)
print("trajectory progress strength (SD of per-second means / SD of speed):", progress_strength)

saved report/C/fig_eda_trajectory_progress.png
trajectory progress strength (SD of per-second means / SD of speed): 0.034


In [21]:
# EDA 5: heterogeneity across vehicles: mean speed per vehicle, sorted
per_unit = df.groupby(UNIT_COL)[TARGET_COL].agg(["mean", "std", "min", "max", "count"]).sort_values("mean")
per_unit.to_csv("report/C/table_per_unit.csv")
fig, ax = plt.subplots(figsize=(FIG_W, 2.0))
ax.fill_between(range(len(per_unit)), per_unit["mean"], color=C_BASE, linewidth=0)
ax.plot(range(len(per_unit)), per_unit["mean"], color=C_MID, lw=0.8)
zero_share = float((per_unit["mean"] < 0.5).mean())
ax.text(
    0.02 * len(per_unit),
    per_unit["mean"].max() * 0.6,
    f"{100 * zero_share:.1f}% of vehicles\naverage below 0.5 m/s",
)
ax.set_xlabel("Vehicles sorted by mean speed")
ax.set_ylabel("Mean speed (m/s)")
savefig(fig, "fig_eda_units")
p05, p95 = per_unit["mean"].quantile([0.05, 0.95])
print("5th and 95th percentile of vehicle mean speed:", round(p05, 2), round(p95, 2))

saved report/C/fig_eda_units.png
5th and 95th percentile of vehicle mean speed: 0.0 12.73


In [22]:
# EDA 6: relation with covariates: decile means and Spearman correlation (robust to outliers)
from scipy.stats import spearmanr
rows = []
fig, axes = plt.subplots(2, 2, figsize=(FIG_W, 3.4))
for ax, c, lab in zip(axes.ravel(), [c for c in EXOG_COLS if c in df.columns], ["(a)", "(b)", "(c)", "(d)"]):
    ok = df[[c, TARGET_COL]].dropna()
    rho, p = spearmanr(ok[c], ok[TARGET_COL])
    rows.append([c, round(rho, 3), p, len(ok)])
    b = ok.groupby(pd.qcut(ok[c], 10, duplicates="drop"), observed=True)[TARGET_COL].mean()
    if len(b) > 1:
        ax.plot(range(1, len(b) + 1), b.values, marker="o", ms=3, color=C_MID)
    else:
        ax.text(0.5, 0.5, "too few distinct values", transform=ax.transAxes, ha="center")
    ax.set_xlabel(f"{NICE[c]}, decile")
    ax.set_ylabel("Mean speed (m/s)")
    panel(ax, f"{lab}  Spearman rho = {rho:.2f}")
savefig(fig, "fig_eda_covariates")
corr_tbl = pd.DataFrame(rows, columns=["covariate", "spearman_rho", "p_value", "n"])
corr_tbl.to_csv("report/C/table_covariate_corr.csv", index=False)
display(corr_tbl)

saved report/C/fig_eda_covariates.png


,covariate,spearman_rho,p_value,n
0,acc,0.043,1.360746e-44,108000
1,yaw_rate,0.009,3.608319e-03,108000
2,n_agents,-0.407,0.000000e+00,109200
3,length,0.134,0.000000e+00,109200


In [23]:
# EDA 7: cross-correlation with lags: does the covariate lead speed? (positive lag = covariate earlier)
def cross_corr(g, x, y, lags):
    return [g[y].corr(g[x].shift(k)) for k in lags]
lags = list(range(0, 31, 5))
lag_seconds = np.array(lags) / 10.0
CC_BEST = {}
fig, ax = plt.subplots(figsize=(FIG_W, 2.0))
for c, col in zip(["acc", "yaw_rate"], [C_MID, C_GREY]):
    cc = np.nanmean(
        [
            cross_corr(g.sort_values(TIME_COL), c, TARGET_COL, lags)
            for _, g in df.groupby(UNIT_COL)
            if len(g) > max(lags)
        ],
        axis=0,
    )
    best_i = int(np.nanargmax(np.abs(cc)))
    CC_BEST[c] = (float(lag_seconds[best_i]), float(cc[best_i]))
    ax.plot(lag_seconds, cc, marker="o", ms=3, color=col)
    ax.text(lag_seconds[-1] + 0.08, cc[-1], NICE[c].split(" (")[0], va="center")
    print(c, "best lag:", round(lag_seconds[best_i], 1), "s | corr:", round(float(cc[best_i]), 3))
ax.axhline(0, color=C_TEXT, lw=0.5, ls="--")
ax.set_xlim(-0.1, 4.1)
ax.set_xlabel("Lag (s), covariate leads speed")
ax.set_ylabel("Mean correlation")
savefig(fig, "fig_eda_crosscorr")

acc best lag: 0.0 s | corr: 0.145
yaw_rate best lag: 0.0 s | corr: 0.014
saved report/C/fig_eda_crosscorr.png


In [24]:
# EDA 8: autocorrelation of speed averaged over trajectories: choose useful history lags in seconds
from statsmodels.tsa.stattools import acf
nl = 30
acfs = [
    acf(g.sort_values(TIME_COL)[TARGET_COL].values, nlags=nl, fft=True)
    for _, g in df.groupby(UNIT_COL)
    if len(g) > nl
]
acf_mean = np.nanmean(acfs, axis=0)  # vehicles with constant speed give NaN and are skipped
lag_seconds = np.arange(nl + 1) / 10.0
fig, ax = plt.subplots(figsize=(FIG_W, 2.0))
ax.vlines(lag_seconds, 0, acf_mean, color=C_MID, lw=1)
ax.plot(lag_seconds, acf_mean, "o", ms=2.5, color=C_MID)
ax.axhline(0, color=C_TEXT, lw=0.5)
ax.set_xlabel("Lag (s)")
ax.set_ylabel("Mean ACF of speed")
savefig(fig, "fig_eda_acf")
print(
    "ACF at 0.1 s:",
    round(acf_mean[1], 3),
    "| at 1 s:",
    round(acf_mean[10], 3),
    "| at 3 s:",
    round(acf_mean[30], 3),
)

saved report/C/fig_eda_acf.png
ACF at 0.1 s: 0.837 | at 1 s: 0.52 | at 3 s: -0.039


In [25]:
# EDA 9: missingness and speed by scene density (density terciles defined exactly as in Notebook A)
miss_cols = df[EXOG_COLS].isna().mean()
first_step_only = df.loc[df.groupby(UNIT_COL).cumcount() > 0, EXOG_COLS].isna().sum().sum() == 0
print(
    "missing share per column:",
    miss_cols.round(4).to_dict(),
    "| missing only at the first step:",
    bool(first_step_only),
)
d9 = df.dropna(subset=["yaw_rate"]).copy()  # Notebook A uses the 108,000 rows with a yaw rate
q33, q66 = d9["n_agents"].quantile([1 / 3, 2 / 3])
d9["density_group"] = pd.Categorical(
    np.select([d9["n_agents"] <= q33, d9["n_agents"] <= q66], ["Low", "Medium"], default="High"),
    categories=["Low", "Medium", "High"],
    ordered=True,
)
density_speed = d9.groupby("density_group", observed=True)[TARGET_COL].mean()
fig, ax = plt.subplots(figsize=(FIG_W * 0.6, 2.0))
bars = ax.bar(
    density_speed.index.astype(str),
    density_speed.values,
    width=0.6,
    color=[C_ACCENT, C_BASE, C_BASE],
    edgecolor=C_TEXT,
    linewidth=0.4,
)
ax.bar_label(bars, labels=[f"{v:.1f}" for v in density_speed.values], padding=1.5)
ax.set_ylim(0, density_speed.max() * 1.22)
ax.set_xlabel("Scene density")
ax.set_ylabel("Mean speed (m/s)")
savefig(fig, "fig_eda_regime")
print(f"density cut points: {q33:.0f} and {q66:.0f} agents")
print(density_speed.round(3))

missing share per column: {'acc': 0.011, 'yaw_rate': 0.011, 'n_agents': 0.0, 'length': 0.0} | missing only at the first step: True
saved report/C/fig_eda_regime.png
density cut points: 36 and 61 agents
density_group
Low       4.466
Medium    2.002
High      0.899
Name: speed, dtype: float64


### Ghi nhận xét EDA (bắt buộc, dùng cho Results RQ1)

Điền 5 nhận xét; mỗi nhận xét phải gắn với một con số hoặc bảng/hình phía trên.

Ví dụ phù hợp với WOMD:

> "Mean speed thay đổi X m/s giữa đầu và cuối trajectory progress (Hình EDA 4), cho thấy trạng thái chuyển động phụ thuộc vào vị trí tương đối trong đoạn quan sát."

Không diễn giải timestamp giả lập thành giờ/ngày/tháng hay mùa lịch.


In [26]:
# EDA 10: five evidence-backed observations, written from the numbers computed above (never typed by hand)
rho_density = corr_tbl.set_index("covariate").loc["n_agents", "spearman_rho"]
eda_notes = [
    f"1. Speed is right-skewed (skewness {df[TARGET_COL].skew():.2f}); {100 * (df[TARGET_COL] == 0).mean():.1f}% of rows have speed 0.",
    f"2. Mean speed differs strongly between vehicles: the 5th and 95th percentiles of vehicle mean speed are {p05:.2f} and {p95:.2f} m/s.",
    f"3. Scene density is negatively associated with speed: Spearman rho(n_agents, speed) = {rho_density:.3f}.",
    f"4. Acceleration correlates most with speed at lag {CC_BEST['acc'][0]:.1f} s (r = {CC_BEST['acc'][1]:.3f}); yaw rate is weakly related (best r = {CC_BEST['yaw_rate'][1]:.3f}).",
    f"5. Mean speed falls from {density_speed['Low']:.3f} m/s in low-density scenes to {density_speed['Medium']:.3f} m/s (medium) and {density_speed['High']:.3f} m/s (high).",
]
open("report/C/eda_notes.md", "w", encoding="utf-8").write("\n".join(eda_notes))
print("\n".join(eda_notes))
print("saved report/C/eda_notes.md")

1. Speed is right-skewed (skewness 2.18); 55.8% of rows have speed 0.
2. Mean speed differs strongly between vehicles: the 5th and 95th percentiles of vehicle mean speed are 0.00 and 12.73 m/s.
3. Scene density is negatively associated with speed: Spearman rho(n_agents, speed) = -0.407.
4. Acceleration correlates most with speed at lag 0.0 s (r = 0.145); yaw rate is weakly related (best r = 0.014).
5. Mean speed falls from 4.466 m/s in low-density scenes to 2.002 m/s (medium) and 0.899 m/s (high).
saved report/C/eda_notes.md


## Bước 3: SQL với DuckDB, kiểm chéo RQ1

**Vì sao SQL**: người đọc bài kiểm tra được từng con số bằng một truy vấn.

- **Q2:** dùng đúng cách chia mật độ của Notebook A: phân vị 1/3 và 2/3, trên các dòng có yaw rate.
- **Q3:** giữ nhóm 0 (không quay đầu, gồm cả xe đứng yên) tách khỏi 10 nhóm theo cường độ rẽ. Bản cũ bỏ nhóm 0 nên tốc độ trong hình RQ1 lên tới 4 m/s trở lên (góp ý 4.3).

Vì vậy kết quả SQL ở đây phải **khớp** với bảng RQ1 của Notebook A. Bảng đặc trưng và mục tiêu dự báo do Notebook A tạo; Notebook C không tạo lại.

In [27]:
# Step 3a: register the clean table and write the RQ1 queries to sql/C_queries.sql
con.register("clean", df)
SQL_RQ1 = f"""
-- Q1: mean and spread of speed by scene density
SELECT n_agents, AVG({TARGET_COL}) AS mean_speed, STDDEV({TARGET_COL}) AS sd_speed, COUNT(*) AS n
FROM clean
WHERE n_agents IS NOT NULL
GROUP BY n_agents
ORDER BY n_agents;
-- Q2: share of high-speed rows by scene-density tercile (same cut points as Notebook A)
WITH base AS (SELECT * FROM clean WHERE yaw_rate IS NOT NULL),
stats AS (
    SELECT quantile_cont(n_agents, 1.0 / 3) AS q33, quantile_cont(n_agents, 2.0 / 3) AS q67,
           quantile_cont({TARGET_COL}, {EVENT_QUANTILE}) AS speed_thr
    FROM base
)
SELECT CASE WHEN n_agents <= q33 THEN 'Low' WHEN n_agents <= q67 THEN 'Medium' ELSE 'High' END AS density_group,
       AVG(({TARGET_COL} > speed_thr)::INT) AS event_share, AVG({TARGET_COL}) AS mean_speed, COUNT(*) AS n
FROM base, stats
GROUP BY 1
ORDER BY 1;
-- Q3: mean speed by turning group: 0 = no heading change (incl. stopped), 1-10 = deciles of |yaw rate| among turning rows
WITH t AS (SELECT {TARGET_COL} AS speed, ABS(yaw_rate) AS ay FROM clean WHERE yaw_rate IS NOT NULL),
g AS (
    SELECT speed, 0 AS turning_group FROM t WHERE ay <= {TURN_EPS}
    UNION ALL
    SELECT speed, NTILE(10) OVER (ORDER BY ay) AS turning_group FROM t WHERE ay > {TURN_EPS}
)
SELECT turning_group, AVG(speed) AS mean_speed, MEDIAN(speed) AS median_speed, COUNT(*) AS n
FROM g
GROUP BY turning_group
ORDER BY turning_group;
-- Q4: Pearson correlations of speed with density and turning
SELECT corr({TARGET_COL}, n_agents) AS corr_speed_density, corr({TARGET_COL}, ABS(yaw_rate)) AS corr_speed_turning
FROM clean
WHERE n_agents IS NOT NULL AND yaw_rate IS NOT NULL;
"""
open("sql/C_queries.sql", "w").write(SQL_RQ1)
print(SQL_RQ1)


-- Q1: mean and spread of speed by scene density
SELECT n_agents, AVG(speed) AS mean_speed, STDDEV(speed) AS sd_speed, COUNT(*) AS n
FROM clean
WHERE n_agents IS NOT NULL
GROUP BY n_agents
ORDER BY n_agents;
-- Q2: share of high-speed rows by scene-density tercile (same cut points as Notebook A)
WITH base AS (SELECT * FROM clean WHERE yaw_rate IS NOT NULL),
stats AS (
    SELECT quantile_cont(n_agents, 1.0 / 3) AS q33, quantile_cont(n_agents, 2.0 / 3) AS q67,
           quantile_cont(speed, 0.9) AS speed_thr
    FROM base
)
SELECT CASE WHEN n_agents <= q33 THEN 'Low' WHEN n_agents <= q67 THEN 'Medium' ELSE 'High' END AS density_group,
       AVG((speed > speed_thr)::INT) AS event_share, AVG(speed) AS mean_speed, COUNT(*) AS n
FROM base, stats
GROUP BY 1
ORDER BY 1;
-- Q3: mean speed by turning group: 0 = no heading change (incl. stopped), 1-10 = deciles of |yaw rate| among turning rows
WITH t AS (SELECT speed AS speed, ABS(yaw_rate) AS ay FROM clean WHERE yaw_rate IS NOT NULL),
g AS (

In [28]:
# Step 3b: run every statement, save each result as report/C/table_q{k}.csv (comment lines removed before splitting on ';')
def run_sql_file(path):
    txt = open(path).read()
    body = "\n".join(l for l in txt.splitlines() if not l.strip().startswith("--"))
    k = 0
    for stmt in body.split(";"):
        stmt = stmt.strip()
        if not stmt:
            continue
        k += 1
        out = con.execute(stmt).df()
        out.to_csv(f"report/C/table_q{k}.csv", index=False)
        print(f"Q{k}: {out.shape}")
        display(out.head(8))
run_sql_file("sql/C_queries.sql")

Q1: (230, 4)


,n_agents,mean_speed,sd_speed,n
0,2,9.054019,1.287223,31
1,3,7.516086,2.821454,14
2,4,8.238541,3.202385,73
3,5,7.915746,5.245847,193
4,6,6.934337,6.840822,346
5,7,9.142120,8.969227,401
6,8,9.507782,8.187507,522
7,9,10.620907,8.574821,704


Q2: (3, 4)


,density_group,event_share,mean_speed,n
0,High,0.030352,0.899248,35648
1,Low,0.195274,4.466138,36308
2,Medium,0.072911,2.001865,36044


Q3: (11, 4)


,turning_group,mean_speed,median_speed,n
0,0,0.001219,0.000000,60951
1,1,4.161917,0.487228,4705
2,2,6.006653,4.447345,4705
3,3,6.328267,4.788846,4705
4,4,6.063347,4.097887,4705
5,5,6.173435,4.695358,4705
6,6,6.189321,4.352814,4705
7,7,6.444303,4.971600,4705


Q4: (1, 2)


,corr_speed_density,corr_speed_turning
0,-0.306979,0.166345


In [29]:
# Step 3c: statistical tests behind RQ1 (report p-values, not only means): Kruskal-Wallis across units, Spearman with covariates
from scipy.stats import kruskal
groups = [g[TARGET_COL].values for _, g in df.groupby(UNIT_COL) if len(g) > 30]
H, p = kruskal(*groups) if len(groups) > 1 else (np.nan, np.nan)
tests = [["Kruskal-Wallis target across units", round(H, 2) if H == H else "n/a", p]]
for c in [c for c in EXOG_COLS if c in df.columns]:
    ok = df[[c, TARGET_COL]].dropna()
    rho, pv = spearmanr(ok[c], ok[TARGET_COL])
    tests.append([f"Spearman target vs {c}", round(rho, 3), pv])
tests = pd.DataFrame(tests, columns=["test", "statistic", "p_value"])
tests.to_csv("report/C/table_rq1_tests.csv", index=False)
display(tests)

,test,statistic,p_value
0,Kruskal-Wallis target across units,107152.210,0.000000e+00
1,Spearman target vs acc,0.043,1.360746e-44
2,Spearman target vs yaw_rate,0.009,3.608319e-03
3,Spearman target vs n_agents,-0.407,0.000000e+00
4,Spearman target vs length,0.134,0.000000e+00


In [30]:
# Step 3d: cross-check with Notebook A (no feature table is built here; Notebook A owns feat.parquet and manifest.json)
man = json.load(open("data/processed/manifest.json"))
origin = pd.read_parquet(man["model_table"]).set_index("unit")
t0 = pd.Timestamp("2000-01-01") + pd.Timedelta(milliseconds=100 * man["origin_step"])
c10 = df[df[TIME_COL] == t0].set_index(UNIT_COL)
common = origin.index.intersection(c10.index)
gap = float(np.abs(c10.loc[common, TARGET_COL] - origin.loc[common, "speed"]).max())
print(
    f"vehicles in Notebook A's modelling table: {len(origin)} | found in Notebook C's clean table: {len(common)}"
)
print(f"max speed difference at the forecast origin (step {man['origin_step']}): {gap:.2e}")
assert len(common) == len(origin) and gap < 1e-9, "Notebook C does not describe the same sample as Notebook A"
a_rq1 = pd.read_csv("report/table_rq1_density.csv").set_index("density_group")
c_rq1 = pd.read_csv("report/C/table_q2.csv").set_index("density_group")
cmp = pd.DataFrame(
    {
        "A_event_share": a_rq1["event_share"],
        "C_event_share": c_rq1["event_share"].round(4),
        "A_mean_speed": a_rq1["mean_speed"],
        "C_mean_speed": c_rq1["mean_speed"].round(4),
    }
)
display(cmp)
assert np.allclose(cmp["A_event_share"], cmp["C_event_share"], atol=1e-3), "SQL RQ1 does not match Notebook A"
print("Notebook C SQL reproduces Notebook A's RQ1 density table.")

vehicles in Notebook A's modelling table: 1172 | found in Notebook C's clean table: 1172
max speed difference at the forecast origin (step 10): 0.00e+00


,A_event_share,C_event_share,A_mean_speed,C_mean_speed
density_group,,,,
High,0.0304,0.0304,0.8992,0.8992
Low,0.1953,0.1953,4.4661,4.4661
Medium,0.0729,0.0729,2.0019,2.0019


Notebook C SQL reproduces Notebook A's RQ1 density table.


In [31]:
# RQ1 figure from the SQL tables (cross-check only: the paper uses Notebook A's Fig. 2)
q2 = pd.read_csv("report/C/table_q2.csv").set_index("density_group").loc[["Low", "Medium", "High"]]
q3 = pd.read_csv("report/C/table_q3.csv")
g0, g1 = q3[q3["turning_group"] == 0].iloc[0], q3[q3["turning_group"] > 0]
fig, axes = plt.subplots(1, 2, figsize=(FIG_W, 2.2), gridspec_kw={"width_ratios": [1, 1.7]})
bars = axes[0].bar(
    q2.index,
    100 * q2["event_share"],
    width=0.6,
    color=[C_ACCENT, C_BASE, C_BASE],
    edgecolor=C_TEXT,
    linewidth=0.4,
)
axes[0].bar_label(bars, labels=[f"{v:.1f}%" for v in 100 * q2["event_share"]], padding=1.5)
axes[0].set_ylim(0, 100 * q2["event_share"].max() * 1.22)
axes[0].set_xlabel("Scene density")
axes[0].set_ylabel("High-speed rows (%)")
panel(axes[0], "(a)")
x0 = -0.8
axes[1].plot(g1["turning_group"], g1["mean_speed"], color=C_TEXT, marker="o", ms=3, lw=1)
axes[1].plot(g1["turning_group"], g1["median_speed"], color=C_MID, marker="^", ms=3, lw=1, ls="--")
axes[1].plot([x0], [g0["mean_speed"]], color=C_TEXT, marker="s", ms=4, ls="none")
axes[1].axvline(0.1, color=C_TEXT, lw=0.4, ls=":")
axes[1].set_xticks([x0] + list(range(1, 11)))
axes[1].set_xticklabels(["0"] + [str(k) for k in range(1, 11)])
axes[1].set_ylim(0, max(q3["mean_speed"].max(), q3["median_speed"].max()) * 1.15)
axes[1].set_xlabel("Turning group (|yaw rate| decile)")
axes[1].set_ylabel("Speed (m/s)")
panel(axes[1], "(b)")
savefig(fig, "fig_rq1_sql")

saved report/C/fig_rq1_sql.png


## Kiểm tra và ghi Audit Log

Notebook_C **không bàn giao dữ liệu**: `feat.parquet` và `manifest.json` thuộc Notebook A. Các test dưới kiểm tra bảng sạch của C. Ô tóm tắt ghi danh sách output trong `report/C/`. Sau đó điền **AI Audit Log**.

In [32]:
# Test cases for Notebook_C (all asserts must pass)
assert df[[UNIT_COL, TIME_COL]].duplicated().sum() == 0, "duplicate unit-time rows"
assert df[TARGET_COL].notna().all(), "target missing after cleaning"
assert (
    df.groupby(UNIT_COL)[TIME_COL].apply(lambda s: s.is_monotonic_increasing).all()
), "time not sorted within unit"
assert df[UNIT_COL].nunique() == man["units"], "Notebook C must describe the same vehicles as Notebook A"
assert df["yaw_rate"].abs().max() < 2 * np.pi / 0.1 - 1, "yaw rate looks unwrapped"
assert not os.path.exists("report/C/feat.parquet"), "Notebook C must not write a feature table"
print("Tests C: OK")

Tests C: OK


In [33]:
# Notebook C summary (no data handoff; Notebook A owns data/processed/feat.parquet and manifest.json)
summary_c = {
    "group": GROUP,
    "topic": TOPIC,
    "dataset": WOMD_SOURCE["name"],
    "source_file": WOMD_SOURCE["path"],
    "rows": len(df),
    "units": int(df[UNIT_COL].nunique()),
    "freq": FREQ,
    "density_cut_points": [float(q33), float(q66)],
    "outputs": sorted(os.listdir("report/C")),
}
json.dump(summary_c, open("report/C/summary_C.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in summary_c.items() if k != "outputs"}, indent=2, ensure_ascii=False))
print("outputs in report/C:", len(summary_c["outputs"]))

{
  "group": "Nhom 7",
  "topic": "Dự đoán quãng đường xe có khoảng dự báo conformal trên Waymo Open Motion Dataset",
  "dataset": "Waymo Open Motion Dataset v1.2",
  "source_file": "data/processed/womd_v1_2.parquet",
  "rows": 109200,
  "units": 1200,
  "freq": "100ms",
  "density_cut_points": [
    36.0,
    61.0
  ]
}
outputs in report/C: 22


In [34]:
# Record the prompts you actually used this week
audit(
    "Step 2b",
    "step 2b sửa như nào và lí do vì sao sửa như vậy",
    "ChatGPT",
    "AI đề xuất thay điều kiện physical range chung của template bằng điều kiện speed >= 0 để phù hợp với biến mục tiêu speed của Waymo.",
    "Chạy lại Step 2b trên 109200 dòng và kiểm tra cleaning log; không có dòng nào bị loại sai.",
    "Giữ điều kiện speed >= 0 vì speed không thể âm và dữ liệu sau cleaning vẫn giữ đủ 109200 dòng.",
    hallucination=False,
)
audit(
    "EDA 7",
    "những điểm cần sửa trong EDA7",
    "ChatGPT",
    "AI đề xuất bỏ logic seasonal của template, chỉ kiểm tra acc và yaw_rate theo lag từ 0 đến 3 giây và đổi trục lag từ step sang giây.",
    "Chạy EDA 7 trên dữ liệu thật: acc best lag 0.0 s, corr 0.161; yaw_rate best lag 2.0 s, corr -0.014.",
    "Giữ cách phân tích theo giây vì Waymo có tần suất 10 Hz và trajectory chỉ kéo dài vài giây.",
    hallucination=False,
)
audit(
    "Step 3d",
    "step 3d sửa những chỗ nào",
    "ChatGPT",
    "AI đề xuất bỏ calendar/seasonal features của template và tạo feature 1 giây cùng target displacement 3 giây và 8 giây từ x, y.",
    "Chạy Step 3d và kiểm tra kết quả: feat có 109200 dòng, 72,363 target 3s hợp lệ và 12,894 target 8s hợp lệ (73,200 và 13,200 là số vị trí có thể, trước khi lọc theo sai lệch thời gian ±0.05 s); Step 3e xác nhận alignment đều True.",
    "Chọn y_disp_3s và y_disp_8s làm target vì đúng với bài toán dự báo quỹ đạo Waymo và phù hợp với Proposal.",
    hallucination=False,
)
print(pd.read_csv(AUDIT_PATH).tail(3))

audit entry saved: Step 2b
audit entry saved: EDA 7
audit entry saved: Step 3d
         date   group     step                                           prompt     tool                                          ai_output  \
0  2026-10-11  Nhom 7  Step 2b  step 2b sửa như nào và lí do vì sao sửa như vậy  ChatGPT  AI đề xuất thay điều kiện physical range chung...   
1  2026-10-11  Nhom 7    EDA 7                    những điểm cần sửa trong EDA7  ChatGPT  AI đề xuất bỏ logic seasonal của template, chỉ...   
2  2026-10-11  Nhom 7  Step 3d                        step 3d sửa những chỗ nào  ChatGPT  AI đề xuất bỏ calendar/seasonal features của t...   

                                        verified_how                                           decision  hallucination  
0  Chạy lại Step 2b trên 109200 dòng và kiểm tra ...  Giữ điều kiện speed >= 0 vì speed không thể âm...              0  
1  Chạy EDA 7 trên dữ liệu thật: acc best lag 0.0...  Giữ cách phân tích theo giây vì Waymo có tần s...   

## Bài tập mở rộng cho Sinh viên A

Làm ít nhất 3 bài và ghi kết quả vào `report/exercises_C.md`.

1. **Feature group bổ sung trong WOMD v1.2**: thử thêm `heading`, `width`, `vx`, `vy` (nếu có) thành một view thứ ba và kiểm tra missingness/đóng góp.
2. **So sánh tần suất**: gộp trajectory từ 100 ms lên 500 ms và kiểm tra mức mất thông tin chuyển động.
3. **Hai truy vấn SQL thêm**: một truy vấn dùng `QUALIFY`, một truy vấn dùng `WITH/CTE`, đều dựa trên scene density hoặc turning.
4. **Phân rã ngắn hạn**: dùng STL như một phép phân rã tín hiệu trên một trajectory; không diễn giải thành mùa vụ lịch.
5. **Kiểm tra rò rỉ có chủ ý**: tạo một feature sai bằng `LEAD`, so với feature hợp lệ và ghi lại vì sao MAE thấp bất thường không phải là kết quả hợp lệ.


In [35]:
# Exercise 4: STL used only as a short-term signal decomposition for one Waymo trajectory
from statsmodels.tsa.seasonal import STL
period = 10  # 10 steps = 1 second at 100 ms
u0 = df.groupby(UNIT_COL)[TARGET_COL].std().idxmax()  # trajectory with the most speed variation
s = df[df[UNIT_COL] == u0].set_index(TIME_COL)[TARGET_COL].asfreq(FREQ).interpolate(limit=3).dropna()
print("unit:", u0, "| observations:", len(s))
if len(s) > 3 * period:
    stl = STL(s, period=period, robust=True).fit()
    t = (s.index - s.index[0]).total_seconds()
    fig, axes = plt.subplots(3, 1, figsize=(FIG_W, 3.2), sharex=True)
    for ax, comp, lab, pl in zip(
        axes, [stl.trend, stl.seasonal, stl.resid], ["Trend", "1-s cycle", "Residual"], ["(a)", "(b)", "(c)"]
    ):
        ax.plot(t, comp.values, color=C_MID, lw=0.8)
        ax.set_ylabel(lab)
        panel(ax, pl)
    axes[-1].set_xlabel("Elapsed time (s); all panels in m/s")
    savefig(fig, "fig_exercise_stl")
    residual_share = float(stl.resid.var() / s.var())
    residual_acf1 = float(stl.resid.autocorr(lag=1))
    print(
        "residual share of variance:",
        round(residual_share, 3),
        "| residual lag-1 autocorrelation:",
        round(residual_acf1, 3),
    )
else:
    print("trajectory is too short for STL")

unit: 337735562f13c4a1_879 | observations: 91
saved report/C/fig_exercise_stl.png
residual share of variance: 0.001 | residual lag-1 autocorrelation: 0.322


In [36]:
# Exercise 2: compare 100 ms working frequency with coarser 500 ms frequency
fine = df[[UNIT_COL, TIME_COL, TARGET_COL]].copy()
fine["elapsed_s"] = (fine[TIME_COL] - fine.groupby(UNIT_COL)[TIME_COL].transform("min")).dt.total_seconds()
fine["k"] = (fine["elapsed_s"] / 0.1).round().astype(int)
coarse = df.set_index(TIME_COL).groupby(UNIT_COL)[TARGET_COL].resample("500ms").mean().reset_index()
coarse["elapsed_s"] = (
    coarse[TIME_COL] - coarse.groupby(UNIT_COL)[TIME_COL].transform("min")
).dt.total_seconds()
coarse["k"] = (coarse["elapsed_s"] / 0.5).round().astype(int)
def progress_strength(frame):
    profile = frame.groupby("k")[TARGET_COL].mean()
    return round(profile.std() / frame[TARGET_COL].std(), 3)
strength_100ms = progress_strength(fine)
strength_500ms = progress_strength(coarse)
print("trajectory progress strength:")
print("100 ms:", strength_100ms)
print("500 ms:", strength_500ms)
print("rows 100 ms:", len(fine))
print("rows 500 ms:", len(coarse))

trajectory progress strength:
100 ms: 0.027
500 ms: 0.03
rows 100 ms: 109200
rows 500 ms: 22800


In [37]:
# Exercise 3: two additional SQL queries for Waymo
# Query 1 uses QUALIFY
# Query 2 uses WITH (CTE)
ex3_q1 = """
WITH base AS (
    SELECT
        unit,
        speed,
        CASE
            WHEN n_agents <= 10 THEN 'Low'
            WHEN n_agents <= 20 THEN 'Medium'
            ELSE 'High'
        END AS density_group
    FROM clean
)
SELECT
    density_group,
    unit,
    AVG(speed) AS mean_speed,
    RANK() OVER (
        PARTITION BY density_group
        ORDER BY AVG(speed) DESC
    ) AS rk
FROM base
GROUP BY density_group, unit
QUALIFY rk <= 5
ORDER BY density_group, rk;
"""
ex3_q2 = """
WITH threshold AS (
    SELECT quantile_cont(speed, 0.9) AS speed_thr
    FROM clean
),
base AS (
    SELECT
        speed,
        CASE
            WHEN n_agents <= 10 THEN 'Low'
            WHEN n_agents <= 20 THEN 'Medium'
            ELSE 'High'
        END AS density_group
    FROM clean
)
SELECT
    density_group,
    AVG((speed > speed_thr)::INT) AS high_speed_share,
    COUNT(*) AS n
FROM base, threshold
GROUP BY density_group
ORDER BY density_group;
"""
ex3_r1 = con.execute(ex3_q1).df()
ex3_r2 = con.execute(ex3_q2).df()
print("Query 1 - Top 5 units by density group")
display(ex3_r1)
print("Query 2 - High-speed share by density group")
display(ex3_r2)

Query 1 - Top 5 units by density group


,density_group,unit,mean_speed,rk
0,High,69ec07d751cf08d3_439,27.992386,1
1,High,37c389322a4831f0_1052,22.048844,2
2,High,d786325e3d746549_302,19.631516,3
3,High,afd2ff89139f7af4_2366,19.054175,4
4,High,d0046cb41a0ab87e_1,18.907859,5
5,Low,90457cb2535317b8_2043,23.082653,1
6,Low,84df35cbf57d8a9c_954,22.972076,2
7,Low,89f2775f1a21e722_464,21.794239,3
8,Low,64a91296f13ac341_509,20.984013,4
9,Low,9be58a7bed8ed751_1832,20.945023,5


Query 2 - High-speed share by density group


,density_group,high_speed_share,n
0,High,0.070042,96300
1,Low,0.437831,3024
2,Medium,0.288680,9876


In [38]:
# Save Exercise 3 SQL queries to sql/C_queries.sql
marker = "-- EXERCISE 3 - WAYMO"
with open("sql/C_queries.sql", "r", encoding="utf-8") as f:
    current_sql = f.read()
if marker not in current_sql:
    with open("sql/C_queries.sql", "a", encoding="utf-8") as f:
        f.write("\n\n")
        f.write(marker + "\n")
        f.write("""
-- Query 1: QUALIFY - Top 5 trajectories by mean speed
-- within each scene-density group
""")
        f.write(ex3_q1.strip())
        f.write("\n\n")
        f.write("""
-- Query 2: WITH / CTE - High-speed event share
-- by scene-density group
""")
        f.write(ex3_q2.strip())
        f.write("\n")
    print("Exercise 3 SQL saved to sql/C_queries.sql")
else:
    print("Exercise 3 SQL already exists in sql/C_queries.sql")

Exercise 3 SQL saved to sql/C_queries.sql


In [39]:
# Save Exercise 2, 3, 4 results to report/C/exercises_C.md
with open("report/C/exercises_C.md", "w", encoding="utf-8") as f:
    # EXERCISE 2
    f.write("# Notebook C - Extended Exercises\n\n")
    f.write("## Exercise 2 - Frequency comparison\n\n")
    f.write("- Working frequency: 100 ms\n")
    f.write("- Coarser frequency: 500 ms\n")
    f.write(f"- Trajectory progress strength at 100 ms: {strength_100ms}\n")
    f.write(f"- Trajectory progress strength at 500 ms: {strength_500ms}\n")
    f.write(f"- Rows at 100 ms: {len(fine)}\n")
    f.write(f"- Rows at 500 ms: {len(coarse)}\n\n")
    f.write("### Conclusion\n")
    f.write(
        "The 100 ms frequency preserves finer vehicle-motion information, "
        "while the 500 ms frequency reduces the number of observations and "
        "smooths short-term variation. The 100 ms frequency is therefore "
        "kept for the main Waymo trajectory pipeline.\n\n"
    )
    # EXERCISE 3
    f.write("## Exercise 3 - Additional SQL queries\n\n")
    f.write("### Query 1 - QUALIFY\n")
    f.write("Top 5 trajectories by mean speed were identified " "within each scene-density group.\n\n")
    f.write("```text\n")
    f.write(ex3_r1.to_string(index=False))
    f.write("\n```\n\n")
    f.write("### Query 2 - WITH / CTE\n")
    f.write(
        "The 90th-percentile speed threshold was used to compare "
        "high-speed event shares across scene-density groups.\n\n"
    )
    f.write("```text\n")
    f.write(ex3_r2.to_string(index=False))
    f.write("\n```\n\n")
    # EXERCISE 4
    f.write("## Exercise 4 - Short-term STL signal decomposition\n\n")
    f.write(f"- Unit: {u0}\n")
    f.write(f"- STL period: {period} steps (1 second; signal period, not calendar seasonality)\n")
    f.write(f"- Residual share of variance: {residual_share:.3f}\n")
    f.write(f"- Residual lag-1 autocorrelation: " f"{residual_acf1:.3f}\n\n")
    f.write("### Interpretation\n")
    if abs(residual_acf1) < 0.2:
        f.write(
            "The residual has weak lag-1 autocorrelation, suggesting "
            "that little short-term temporal structure remains after "
            "the STL decomposition.\n"
        )
    else:
        f.write(
            "The residual still has noticeable lag-1 autocorrelation, "
            "suggesting that temporal structure remains after the STL "
            "decomposition and may require additional features or "
            "a more suitable model.\n"
        )
print("saved Exercise 2, 3, 4 to report/C/exercises_C.md")

saved Exercise 2, 3, 4 to report/C/exercises_C.md


### Tự đánh giá Notebook_C

| Tiêu chí | Tự chấm (0-2) | Bằng chứng |
|---|---|---|
| Nguồn WOMD v1.2 và giấy phép rõ ràng, cùng mẫu với Notebook A | | `report/C/data_sources.csv`, ô Step 3d |
| Hai feature views khớp key chính xác | | `report/C/table_feature_view_alignment.csv` |
| Làm sạch có nhật ký | | `report/C/table_cleaning_log.csv` |
| EDA có nhận xét kèm số (số tự tính, không gõ tay) | | `report/C/eda_notes.md` |
| SQL khớp RQ1 của Notebook A | | `sql/C_queries.sql`, ô Step 3d |
| Bài tập mở rộng | | `report/C/exercises_C.md` |
| AI Audit Log | | `report/C/ai_audit_log.csv` |

## Lỗi thường gặp và cách sửa

| Lỗi | Nguyên nhân | Cách sửa |
|---|---|---|
| `KeyError: 'ts'` | tên cột trong file khác CONFIG | in `df.columns`, sửa `TIME_COL` |
| Hai feature views lệch key | extraction của hai file không cùng `[unit, ts]` | dừng pipeline và sửa extraction; không dùng nearest merge để che lỗi |
| Rò rỉ tương lai | dùng `LEAD` hoặc giá trị tương lai làm feature | chỉ dùng `LAG`/rolling quá khứ cho predictors |
| Trùng dòng theo unit-time | extraction tạo bản ghi lặp | `drop_duplicates([UNIT_COL, TIME_COL])` và ghi cleaning log |
| Auxiliary feature thiếu nhiều | state không hợp lệ ở một số timestep | báo missing share; không tự điền bằng giá trị tương lai |
| DuckDB `Binder Error` | tên cột/tên bảng không đúng | in schema và sửa SQL |
| `MemoryError` | nạp file lớn không cần thiết | dùng DuckDB và chọn cột sớm |
| Diễn giải "seasonality" sai | timestamp chỉ là trajectory progress | báo theo seconds/steps, không theo giờ/ngày/tháng |


## Checklist Notebook_C

- [ ] Notebook A đã chạy xong (`womd_v1_2.parquet`, `feat_core_origin10.parquet`, `manifest.json` tồn tại)
- [ ] Restart & Run All không lỗi; số thứ tự chạy tăng liên tục
- [ ] `report/C/table_feature_view_alignment.csv` cho thấy main/auxiliary views khớp `[unit, ts]`
- [ ] Yaw rate đã xử lý góc (không có giá trị quanh ±62.8 rad/s)
- [ ] Ô Step 3d: C mô tả đúng mẫu xe của A và SQL Q2 khớp bảng RQ1 của A
- [ ] Mọi output của C nằm trong `report/C/`; C không ghi `feat.parquet` hay `manifest.json`
- [ ] AI Audit Log được cập nhật bằng các prompt thực tế

## Mẫu viết phần Data và RQ1 cho bài báo

Số trong mục Data (3.2) và RQ1 (5.1) của bài **lấy từ `report/paper_numbers_A.csv`** của Notebook A. Hình RQ1 là Fig. 2 của Notebook A. Notebook C dùng để kiểm chéo bằng SQL. Nếu số của C và A lệch nhau thì dừng lại tìm nguyên nhân trước khi viết.

**Data.** We use the Waymo Open Motion Dataset (WOMD) v1.2 under the Waymo Open Dataset License. Speed, acceleration, yaw rate, vehicle length and scene density are taken from the same 1,200 sampled vehicle trajectories at 10 Hz.

**RQ1.** Speed varies strongly with scene density and between vehicles, while it shows no monotonic trend with turning intensity once rows without heading change (including stopped vehicles) are separated.